# Connect Drive

In [ ]:
from google.colab import drive
drive.mount('./content')

Mounted at ./content


# Libraries

In [ ]:
import os
import torch
import torchvision.transforms as transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
from torch.utils.data import ConcatDataset
import matplotlib.pyplot as plt
import torch.optim as optim
from collections import deque
import numpy as np
import pickle
import sys
import cv2
import gc
import psutil

# Load Images

In [ ]:
unlabel_train_path = '/content/content/MyDrive/MVTec/bottle/train/good'
unlabel_test_path = '/content/content/MyDrive/MVTec/bottle/test/good'
label_path = '/content/content/MyDrive/MVTec/bottle/test'
ground_truth_path = '/content/content/MyDrive/MVTec/bottle/ground_truth'

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30, fill=0),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), fill=0),
    transforms.RandomAffine(degrees=0, shear=20, fill=0),
    #transforms.RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0))
])

def add_gaussian_noise(image, mean=0, std=0.1):
    noise = torch.randn_like(image[:3]) * std + mean
    image[:3] += noise
    return torch.clamp(image, 0., 1.)

In [ ]:
def filtering(input):
    """
    Compute R_clone using Sobel edge detection after applying Gaussian blur on grayscale patches.
    Encourages the model to focus on edges and boundaries.

    Parameters:
        patches: Tensor of shape [batch, channels, height, width]
                 (image patches)

    Returns:
        Scalar tensor representing R_clone.
    """
    # Define Sobel filters
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)

    # Convert to grayscale using weighted sum of RGB channels
    gray_image = 0.2989 * input[:, 0:1, :, :] + 0.5870 * input[:, 1:2, :, :] + 0.1140 * input[:, 2:3, :, :]

    # Apply Gaussian Blur (define a simple 5x5 kernel)
    kernel = torch.tensor([[1, 4, 6, 4, 1], [4, 16, 24, 16, 4], [6, 24, 36, 24, 6], [4, 16, 24, 16, 4], [1, 4, 6, 4, 1]], dtype=torch.float32).view(1, 1, 5, 5) / 256.0
    smoothed_image = F.conv2d(gray_image, kernel.to(input.device), padding=2)

    # Apply Sobel filters to the blurred image
    edges_x = F.conv2d(smoothed_image, sobel_x.to(input.device), padding=1)
    edges_y = F.conv2d(smoothed_image, sobel_y.to(input.device), padding=1)

    # Compute edge intensity
    edge_intensity = torch.sqrt(edges_x**2 + edges_y**2)

    return edge_intensity

In [ ]:
class ImageLoaderDataset(Dataset):
    def __init__(self, directory_path, transform=None, ignore_folders=None, subfolder=True, include_history=True,
                 include_loss_history=True, include_counter=True, padding=True, padding_mode='constant', patch_size=64, crop_size=128, move_size=24, gray=False):
        """
        A custom Dataset class for loading images with optional padding and additional history/loss history layers.

        Parameters:
            directory_path (str): Path to the folder containing images.
            transform (callable, optional): A function/transform to apply to the images.
            ignore_folders (list, optional): Folders to ignore while loading images.
            subfolder (bool, optional): If True, load images from subfolders.
            include_history (bool, optional): If True, adds a binary history layer to each image.
            include_loss_history (bool, optional): If True, adds a loss history layer to each image.
            include_counter (bool, optional): If True, adds a counter layer to each image.
            padding (bool, optional): Whether to add padding to the images.
            padding_mode (str, optional): The type of padding to apply (e.g., 'constant', 'reflect', etc.).
            patch_size (int): The size of the patches to be extracted.
            crop_size (int): The size of the crop/patch extracted initially.
            move_size (int): The size of the movement for the center of the patch.
        """
        self.directory_path = directory_path
        self.transform = transform if transform else transforms.ToTensor()
        self.ignore_folders = ignore_folders if ignore_folders else []
        self.subfolder = subfolder
        self.include_history = include_history
        self.include_loss_history = include_loss_history
        self.include_counter = include_counter  # Added the counter parameter
        self.padding = padding
        self.padding_mode = padding_mode
        self.patch_size = patch_size
        self.crop_size = crop_size
        self.move_size = move_size
        self.gray = gray
        self.group_indices = {}
        self.image_paths = self._load_image_paths()

    def _load_image_paths(self):
        """
        Loads the paths of all images in the directory, including subfolders if specified.

        Returns:
            list: A list of paths to the image files.
        """
        image_paths = []
        if self.subfolder:
            start_idx = 0
            subfolders = sorted(os.listdir(self.directory_path))
            subfolders = [sf for sf in subfolders if sf not in self.ignore_folders]
            print('Sorted Subfloders:')
            print(subfolders)
            for subfolder in subfolders:
                subfolder_path = os.path.join(self.directory_path, subfolder)
                if os.path.isdir(subfolder_path):
                    image_names = sorted(os.listdir(subfolder_path))
                    for img_name in image_names:
                        print(f'name image in subfolder {subfolder_path}')
                        print(img_name)
                        img_path = os.path.join(subfolder_path, img_name)
                        if img_path.endswith((".png", ".jpg", ".jpeg")):
                            image_paths.append(img_path)
                    end_idx = start_idx + len(image_names) - 1
                    self.group_indices[subfolder] = (start_idx, end_idx)
                    start_idx = end_idx + 1
        else:
            image_names = sorted(os.listdir(self.directory_path))
            for img_name in image_names:
                img_path = os.path.join(self.directory_path, img_name)
                if img_path.endswith((".png", ".jpg", ".jpeg")):
                    image_paths.append(img_path)
        return image_paths

    def __len__(self):
        """Returns the number of images in the dataset."""
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
        Loads an image and applies the specified transformations.

        Parameters:
            idx (int): Index of the image to load.

        Returns:
            torch.Tensor: The transformed image tensor.
        """
        img_path = self.image_paths[idx]
        if self.gray:
          img = Image.open(img_path).convert("1") # Binary-Scale for Ground-Truths
        else:
          img = Image.open(img_path).convert("RGB") # RGB for Images
        img_tensor = self.transform(img)

        # Add binary history layer (initially all zeros) if enabled
        if self.include_history:
            history_layer = torch.zeros((1, img_tensor.size(1), img_tensor.size(2)))
            img_tensor = torch.cat([img_tensor, history_layer], dim=0)

        # Add loss history layer (initially all zeros) if enabled
        if self.include_loss_history:
            loss_history_layer = torch.zeros((1, img_tensor.size(1), img_tensor.size(2)))
            img_tensor = torch.cat([img_tensor, loss_history_layer], dim=0)

        # Add counter layer (initially all zeros) if enabled
        if self.include_counter:
            counter_layer = torch.zeros((1, img_tensor.size(1), img_tensor.size(2)))
            img_tensor = torch.cat([img_tensor, counter_layer], dim=0)

        return img_tensor

In [ ]:
unlabel_train = ImageLoaderDataset(directory_path=unlabel_train_path, transform=transform, subfolder=False)
unlabel_test = ImageLoaderDataset(directory_path=unlabel_test_path, transform=transform, subfolder=False)

In [ ]:
label_dataset = ImageLoaderDataset(directory_path=label_path, transform=transform, ignore_folders=['good'],
                                   subfolder=True)

Sorted Subfloders:
['.ipynb_checkpoints', 'broken_large', 'broken_small', 'contamination']
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
000.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
001.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
002.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
003.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
004.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
005.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
006.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
007.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
008.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/test/broken_large
009.png
name image in subfolder /cont

In [ ]:
ground_truth = ImageLoaderDataset(directory_path=ground_truth_path, transform=transform, subfolder=True,
                                  include_history=False, include_loss_history=False, include_counter=False, gray=True)

Sorted Subfloders:
['.ipynb_checkpoints', 'broken_large', 'broken_small', 'contamination']
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
000_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
001_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
002_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
003_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
004_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
005_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
006_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
007_mask.png
name image in subfolder /content/content/MyDrive/MVTec/bottle/ground_truth/broken_large
008_mask.png


In [ ]:
random.seed(42)

label_train = []
groundtruth_train = []
label_test = []
groundtruth_test = []

for group, (start_idx, end_idx) in label_dataset.group_indices.items():
    if group != '.ipynb_checkpoints':
      random_indices = random.sample(range(start_idx, end_idx + 1), 5)
      for idx in random_indices:
        label_train.append(label_dataset[idx])
        groundtruth_train.append(ground_truth[idx])

      remaining_indices = [i for i in range(start_idx, end_idx + 1) if i not in random_indices]
      for idx in remaining_indices:
        label_test.append(label_dataset[idx])
        groundtruth_test.append(ground_truth[idx])

In [ ]:
default_ground_truth = torch.ones(1)

In [ ]:
train_data = list(zip(label_train, groundtruth_train)) + [(data, default_ground_truth) for data in unlabel_train]
test_data = list(zip(label_test, groundtruth_test)) + [(data, default_ground_truth) for data in unlabel_test]

In [ ]:
gc.collect()
os.system('echo 1 > /proc/sys/vm/drop_caches')

512

In [ ]:
random.shuffle(train_data)
random.shuffle(test_data)

In [ ]:
def apply_augmentation(image, label):

    image_rgb = image[:3]
    image_extra = image[3:]

    state = torch.get_rng_state()
    augmented_rgb = augmentation(image_rgb)
    torch.set_rng_state(state)
    augmented_labels = augmentation(label)

    augmented_image = torch.cat([augmented_rgb, image_extra], dim=0)

    del image_rgb, image_extra, augmented_rgb
    gc.collect()
    os.system('echo 1 > /proc/sys/vm/drop_caches')
    return augmented_image, augmented_labels

In [ ]:
final_train = []
num_augmentations = 2

# Assuming train_data is a list of tuples: (image, ground_truth)
for image, label in train_data:
    final_train.append((image, label))

    # Iterate over each (image, ground_truth) pair in train_data
    for _ in range(num_augmentations):
        # Apply augmentation to ground_truth only if it's not a fixed value like tensor([1])
        if not torch.equal(label, torch.tensor([1.0])):  # Check if labels are not tensor([1])
            augmented_image, augmented_labels = apply_augmentation(image, label)
            augmented_image = add_gaussian_noise(augmented_image)
        else:
            augmented_image = augmentation(image)
            augmented_image = add_gaussian_noise(augmented_image)
            augmented_labels = label  # If labels are fixed, keep them unchanged

        # After augmenting all data in this iteration, append the new augmented data pair
        final_train.append((augmented_image, augmented_labels))

In [ ]:
vars_to_delete = [
    'ConcatDataset', 'transform', 'augmentation', 'add_gaussian_noise',
    'ImageLoaderDataset', 'apply_augmentation', 'filtering'
    "label1", "unlabel_train_path", "unlabel_test_path", "label_path",
    "ground_truth_path", "transform", "augmentation", "add_gaussian_noise",
    "unlabel_train", "unlabel_test", "image1", "image2", "label_dataset",
    "user_vars", "augmented_image", "augmented_labels", "image", "labels",
    "num_augmentations", "augmented_imgs_list", "augmented_labels_list",
    "train_data", "default_ground_truth", "remaining_indices", "label_train",
    "groundtruth_train", "label_test", "groundtruth_test", "group",
    "start_idx", "end_idxrandom_indices", "idx", "ground_truth",
]

for var in vars_to_delete:
    if var in globals():
        del globals()[var]
        gc.collect()
        os.system('echo 1 > /proc/sys/vm/drop_caches')

del vars_to_delete
gc.collect()
os.system('echo 1 > /proc/sys/vm/drop_caches')
print('Ok.')

Ok.


In [ ]:
train_data = final_train
del final_train
gc.collect()
os.system('echo 1 > /proc/sys/vm/drop_caches')

512

In [ ]:
print(f"Memory usage: {psutil.virtual_memory().percent}%")

Memory usage: 32.4%


# Neural Batch Sampler Class

In [ ]:
class NeuralBatchSampler(nn.Module):
    def __init__(self, input_size=900, crop_size=128, patch_size=64, move_size=24, action_space=9):
        """
        Initialize the NeuralBatchSampler class with necessary parameters.

        Parameters:
            input_size (int): The size of the input image (height/width).
            crop_size (int): The size of the crop/patch extracted from the image.
            patch_size (int): The size of the patches used in the network.
            move_size (int): The size by which the center of the patch is moved.
            action_space (int): The number of possible actions (directions).
        """
        super(NeuralBatchSampler, self).__init__()
        self.input_size = input_size
        self.patch_size = patch_size
        self.crop_size = crop_size
        self.move_size = move_size
        self.action_space = action_space

        # Define convolutional layers for feature extraction
        self.conv1 = nn.Conv2d(6, 16, kernel_size=3, stride=2, padding=1)  # First convolution layer (5 input channels)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1)
        self.conv4 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv5 = nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1)

        # Fully connected layers for final classification/output
        self.fc1 = nn.Linear(64 * 4 * 4, 256)  # Linear layer after convolutional layers (flattened)
        self.fc2 = nn.Linear(256, action_space)  # Output layer with number of actions

        # Batch normalization layers to improve training stability
        self.bn1 = nn.BatchNorm2d(16)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(32)
        self.bn4 = nn.BatchNorm2d(64)
        self.bn5 = nn.BatchNorm2d(64)

    def forward(self, patches):
        """
        Forward pass for the model, performing feature extraction and classification.

        Parameters:
            patches (Tensor): The image patches to be passed through the network.
            num_episode (int): The current episode number, used to change the input channels.

        Returns:
            Tensor: The predicted action probabilities from the network (softmax output).
        """
        # Forward pass through convolutional layers with ReLU activations and batch normalization
        #patches = patches.permute(0, 3, 1, 2) # Assume input might be (batch_size, height, width, channels)
        x = self.conv1(patches)
        x = F.relu(x)
        x = self.bn1(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = self.bn2(x)

        x = self.conv3(x)
        x = F.relu(x)
        x = self.bn3(x)

        x = self.conv4(x)
        x = F.relu(x)
        x = self.bn4(x)

        x = self.conv5(x)
        x = F.relu(x)
        x = self.bn5(x)

        # Flatten the feature map and pass through fully connected layers
        x = torch.flatten(x, start_dim=1)
        #print("Flattened shape:", x.shape)
        x = F.relu(self.fc1(x))  # First fully connected layer with ReLU activation
        x = F.softmax(self.fc2(x), dim=1)  # Output layer with softmax activation to get action probabilities

        return x
    '''
    def select_crops(self, images, centers=None):
        """
        Select crops from the images based on the specified centers. If no centers are provided,
        they will be randomly generated within the valid range.

        Parameters:
            images (Tensor): The batch of input images from which patches will be extracted.
            centers (Tensor, optional): The center coordinates of the patches. If None, will be randomly generated.

        Returns:
            Tuple: A tuple containing:
                - patches (Tensor): The extracted image patches.
                - centers (Tensor): The center coordinates used for extracting the patches.
        """
        # Use no_grad since we don't need gradient computation here
        with torch.no_grad():
          batch_size, channels, _, _ = images.shape
          height, width = self.input_size, self.input_size
          # If no centers are provided, randomly generate them within valid bounds
          #if centers is None:
              #x_centers = torch.randint(self.crop_size // 2, height - self.crop_size // 2, (batch_size,))
              #y_centers = torch.randint(self.crop_size // 2, width - self.crop_size // 2, (batch_size,))
              #centers = torch.stack([x_centers, y_centers], dim=1)

          # Calculate the starting and ending coordinates for each patch
          x_starts = centers[:, 0] - self.crop_size // 2
          y_starts = centers[:, 1] - self.crop_size // 2
          x_ends = x_starts + self.crop_size
          y_ends = y_starts + self.crop_size


        crops = torch.zeros((batch_size, channels, self.crop_size, self.crop_size), device=images.device)
        for i in range(batch_size):
            # Ensure coordinates are within bounds
            x_start = max(0, x_starts[i])
            y_start = max(0, y_starts[i])
            x_end = min(width, x_ends[i])
            y_end = min(height, y_ends[i])

            crop = images[i, :, y_start:y_end, x_start:x_end]
            crops[i, :, :crop.shape[1], :crop.shape[2]] = crop
            del crop

        return crops, centers
    '''
    def select_crops(self, image, center=None):
      """
      Select a crop from the image based on the specified center. If no center is provided,
      it will be randomly generated within the valid range.

      Parameters:
          image (Tensor): The input image from which the patch will be extracted.
          center (Tensor, optional): The center coordinates of the patch. If None, will be randomly generated.

      Returns:
          Tuple: A tuple containing:
              - patch (Tensor): The extracted image patch.
              - center (Tensor): The center coordinates used for extracting the patch.
      """
      # Use no_grad since we don't need gradient computation here
      with torch.no_grad():
        batch_size, channels, _, _ = image.shape
        height, width = self.input_size, self.input_size
        '''
        if center is None:
            # If no center is provided, randomly generate one within valid bounds
            center_x = torch.randint(self.crop_size // 2, width - self.crop_size // 2, (1,))
            center_y = torch.randint(self.crop_size // 2, height - self.crop_size // 2, (1,))
            center = torch.stack([center_x, center_y], dim=0)
        '''
        center[0] = int(center[0].item())
        center[1] = int(center[1].item())
        if (center[0] - (self.crop_size // 2)) < 0:
          center[0] = center[0] + (self.crop_size // 2 - center[0])
        elif (center[0] + self.crop_size // 2) > height:
          c = height - center[0]
          cc = (self.crop_size // 2) - c
          center[0] = center[0] - cc

        if (center[1] - (self.crop_size // 2)) < 0:
          center[1] = center[1] + (self.crop_size // 2 - center[1])
        elif (center[1] + self.crop_size // 2) > width:
          c = width - center[1]
          cc = (self.crop_size // 2) - c
          center[1] = center[1] - cc

        # Calculate the starting and ending coordinates for the crop
        x_start = int(center[0] - self.crop_size // 2)
        y_start = int(center[1] - self.crop_size // 2)
        x_end = x_start + self.crop_size
        y_end = y_start + self.crop_size

        # Extract the patch from the image
        patch = image[:, :, y_start:y_end, x_start:x_end]

        return patch, center

    '''
    def move_select_patches(self, centers, actions, images):
        """
        Move the selected patches based on the given actions. The patches are moved by a specified move size
        in the image and the history layer is updated accordingly.

        Parameters:
            centers (Tensor): The current centers of the patches.
            actions (Tensor): The actions specifying the movement direction for each patch.
            images (Tensor): The input batch of images.

        Returns:
            Tuple: A tuple containing:
              - move_mask (Tensor): A binary mask indicating which patches were moved (shape: [batch_size]).
              - rows (Tensor): Row indices used for extracting patches (shape: [num_moved_patches, patch_size]).
              - cols (Tensor): Column indices used for extracting patches (shape: [num_moved_patches, patch_size]).
              - patches (Tensor): The extracted image patches after movement (shape: [num_moved_patches, channels, patch_size, patch_size]).
        """
        # Use no_grad since we don't need gradient computation here
        with torch.no_grad():
          batch_size, channels, ـ, ـ = images.shape
          height, width = self.input_size, self.input_size
          """
          Define the movement directions (dx, dy) for the actions:
            Action 1: Move up-left    -> dx = , dy =
            Action 2: Move down-left  -> dx = -move_size, dy =
            Action 3: Move up-right   -> dx = +move_size, dy = 0
            Action 4: Move down-right -> dx = +move_size, dy = 0
            Action 5: Stay (no movement) -> dx = 0, dy = 0
            Action 6: Move left       -> dx = -move_size, dy = +move_size
            Action 7: Move right      -> dx = +move_size, dy = +move_size
            Action 8: Move down-left  -> dx = -move_size, dy = -move_size
            Action 9: Move down-right -> dx = +move_size, dy = -move_size
          """
          dx = torch.tensor([float('nan'), -self.move_size, self.move_size, 0, 0, -self.move_size, -self.move_size, self.move_size, self.move_size])
          dy = torch.tensor([float('nan'), 0, 0, -self.move_size, self.move_size, -self.move_size, self.move_size, -self.move_size, self.move_size])

          if not isinstance(actions, torch.Tensor):
              actions = torch.tensor(actions)

          actions = actions.long()
          # Create a mask for which patches will be moved (i.e., actions != 0)
          move_mask = actions != 0
          new_x = centers[:, 0].clone()
          new_x = new_x.float()
          new_y = centers[:, 1].clone()
          new_y = new_y.float()

          # Update the centers based on the actions
          new_x[move_mask] += dx[actions[move_mask]]
          new_y[move_mask] += dy[actions[move_mask]]

          # Clamp the new centers within the valid range
          new_x[~move_mask] = float('nan')
          new_y[~move_mask] = float('nan')
          new_x = new_x.clamp(0, self.input_size - 1)
          new_y = new_y.clamp(0, self.input_size - 1)

          # Stack the new x and y coordinates to form the new centers
          new_centers = torch.stack([new_x, new_y], dim=1)

          patches = []
          half_patch = self.patch_size // 2
          for i in range(batch_size):
            if not move_mask[i]:
              continue

            x_start = int(new_x[i].item()) - half_patch
            y_start = int(new_y[i].item()) - half_patch
            x_end = x_start + self.patch_size
            y_end = y_start + self.patch_size

            # Extract patch
            patch = images[i, :3, y_start:y_end, x_start:x_end]
            patches.append(patch)

            # Update the history layer
            images[i, 3, y_start:y_end, x_start:x_end] = 1

        # Convert patches to a tensor
        patches = torch.stack(patches) if patches else torch.empty(0, 3, self.patch_size, self.patch_size, device=images.device)
        return move_mask, new_centers, patches
    '''
    def move_select_patches(self, center, action, image):
      """
      Move the selected patch based on the given action. The patch is moved by a specified move size
      in the image and the history layer is updated accordingly.

      Parameters:
          center (Tensor): The current center of the patch (shape: [2]).
          action (int): A single action specifying the movement direction for the patch.
          image (Tensor): The input image (shape: [channels, height, width]).

      Returns:
          Tuple: A tuple containing:
            - move_mask (Tensor): A binary mask indicating if the patch was moved (scalar).
            - new_center (Tensor): The updated center of the patch (shape: [2]).
            - patch (Tensor): The extracted image patch after movement (shape: [channels, patch_size, patch_size]).
      """
      with torch.no_grad():
        batch_size, channels, _, _ = image.shape
        height, width = self.input_size, self.input_size
        # Define the movement directions (dx, dy) for the actions
        dx = torch.tensor([float('nan'), -self.move_size, self.move_size, 0, 0, -self.move_size, -self.move_size, self.move_size, self.move_size])
        dy = torch.tensor([float('nan'), 0, 0, -self.move_size, self.move_size, -self.move_size, self.move_size, -self.move_size, self.move_size])

        # Clone the current center and convert to float
        new_x = center[0].clone().float()
        new_y = center[1].clone().float()

        new_x += dx[action]
        new_y += dy[action]
        # Clamp the new center within the valid range of the image
        new_x = torch.clamp(new_x, min=self.crop_size // 2, max=width - self.crop_size // 2)
        new_y = torch.clamp(new_y, min=self.crop_size // 2, max=height - self.crop_size // 2)


        # Stack the new x and y coordinates to form the new center
        new_center = torch.tensor([new_x, new_y])

        # Extract the patch based on the new center
        half_patch = self.patch_size // 2
        x_start = int(new_x) - half_patch
        y_start = int(new_y) - half_patch
        x_end = x_start + self.patch_size
        y_end = y_start + self.patch_size

        patch = image[:, :3, y_start:y_end, x_start:x_end]

        # Update the history layer
        image[:, 3, y_start:y_end, x_start:x_end] = 1  # Assuming history layer is the 4th channel

        mask_inside_patch = torch.zeros_like(image[:, 5, :, :])  # Create a mask of all ones
        mask_inside_patch[:, y_start:y_end, x_start:x_end] = 1  # Set patch area to 1

        # Add 1 to the counter layer for the pixels inside the patch
        image[:, 5, :, :] += mask_inside_patch

        del mask_inside_patch
        gc.collect()
        torch.cuda.empty_cache()

        patch_channel_6 = image[:, 5, y_start:y_end, x_start:x_end]

        return patch, patch_channel_6, new_center

    def initial_patches(self, images):
        """
        Randomly initialize patch centers and extract patches from the images.

        Parameters:
            images (Tensor): The input batch of images (shape: [batch_size, channels, height, width]).

        Returns:
            Tuple:
                - patches (Tensor): The extracted patches of shape [batch_size, channels, patch_size, patch_size].
                - centers (Tensor): The randomly selected centers of the patches (shape: [batch_size, 2]).
        """
        # Use no_grad since we don't need gradient computation here
        with torch.no_grad():
          batch_size, channels, height, width = images.shape

          # Generate random centers within valid bounds
          x_centers = torch.randint(self.crop_size // 2, width - self.crop_size // 2, (batch_size,), device=images.device)
          y_centers = torch.randint(self.crop_size // 2, height - self.crop_size // 2, (batch_size,), device=images.device)
          centers = torch.stack([x_centers, y_centers], dim=1)


          x_starts = centers[:, 0] - self.patch_size // 2
          y_starts = centers[:, 1] - self.patch_size // 2
          x_ends = x_starts + self.patch_size
          y_ends = y_starts + self.patch_size
          # Initialize tensor for patches
          patches = torch.zeros((batch_size, 3, self.patch_size, self.patch_size), device=images.device)
          patches_channel_6 = torch.zeros((batch_size, 1, self.patch_size, self.patch_size), device=images.device)
          # Extract patches based on the random centers
          for i in range(batch_size):
              # Extract the patch and assign it to the corresponding position
              patch = images[i, :3, y_starts[i]:y_ends[i], x_starts[i]:x_ends[i]]
              patches[i] = patch
              del patch
              patche_channel_6 = images[i, 5, y_starts[i]:y_ends[i], x_starts[i]:x_ends[i]]
              patches_channel_6[i] = patche_channel_6
              del patche_channel_6
              gc.collect()
              torch.cuda.empty_cache()

          return patches, patches_channel_6, centers

# AutoEncoder Class

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, K=200):
        """
        Autoencoder with 8 convolutional and 8 deconvolutional layers,
        including a shortcut connection for speeding up training.

        Parameters:
            K (int): Number of bottleneck channels.
        """
        super(Autoencoder, self).__init__()

        # Encoder
        self.conv1 = nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(64, affine=False)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64, affine=False)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128, affine=False)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(128, affine=False)
        self.conv5 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.bn5 = nn.BatchNorm2d(256, affine=False)
        self.conv6 = nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1)
        self.bn6 = nn.BatchNorm2d(128, affine=False)
        self.conv7 = nn.Conv2d(128, 64, kernel_size=3, stride=1, padding=1)
        self.bn7 = nn.BatchNorm2d(64, affine=False)
        self.conv8 = nn.Conv2d(64, K, kernel_size=8, stride=1, padding=0)

        # Decoder
        self.deconv1 = nn.ConvTranspose2d(K, 64, kernel_size=8, stride=1, padding=0)
        self.bn8 = nn.BatchNorm2d(64, affine=False)
        self.deconv2 = nn.ConvTranspose2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn9 = nn.BatchNorm2d(128, affine=False)
        self.deconv3 = nn.ConvTranspose2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn10 = nn.BatchNorm2d(256, affine=False)
        self.deconv4 = nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1)
        self.bn11 = nn.BatchNorm2d(256, affine=False)
        self.deconv5 = nn.ConvTranspose2d(256, 128, kernel_size=3, stride=1, padding=1)
        self.bn12 = nn.BatchNorm2d(128, affine=False)
        self.deconv6 = nn.ConvTranspose2d(128, 128, kernel_size=4, stride=2, padding=1)
        self.bn13 = nn.BatchNorm2d(128, affine=False)
        self.deconv7 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=1, padding=1)
        self.bn14 = nn.BatchNorm2d(64, affine=False)
        self.deconv8 = nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        """
        Forward pass for the autoencoder.

        Parameters:
            x (Tensor): Input image tensor.

        Returns:
            Tensor: Reconstructed image tensor.
        """

        # Ensure input dimensions are in (batch_size, channels, height, width)
        if x.shape[1] != 3:  # Assume input might be (batch_size, height, width, channels)
            x = x.permute(0, 3, 1, 2)

        # Encoder
        x1 = F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.2)
        x2 = F.leaky_relu(self.bn2(self.conv2(x1)), negative_slope=0.2)
        x3 = F.leaky_relu(self.bn3(self.conv3(x2)), negative_slope=0.2)
        x4 = F.leaky_relu(self.bn4(self.conv4(x3)), negative_slope=0.2)
        x5 = F.leaky_relu(self.bn5(self.conv5(x4)), negative_slope=0.2)
        x6 = F.leaky_relu(self.bn6(self.conv6(x5)), negative_slope=0.2)
        x7 = F.leaky_relu(self.bn7(self.conv7(x6)), negative_slope=0.2)
        bottleneck = self.conv8(x7)

        # Decoder
        d1 = F.leaky_relu(self.bn8(self.deconv1(bottleneck)), 0.2)
        d2 = F.leaky_relu(self.bn9(self.deconv2(d1)), 0.2)
        d3 = F.leaky_relu(self.bn10(self.deconv3(d2)), 0.2)
        d3 = torch.cat((d3, x5), dim=1)  # Skip connection
        d4 = F.leaky_relu(self.bn11(self.deconv4(d3)), 0.2)
        d5 = F.leaky_relu(self.bn12(self.deconv5(d4)), 0.2)
        d6 = F.leaky_relu(self.bn13(self.deconv6(d5)), 0.2)
        d7 = F.leaky_relu(self.bn14(self.deconv7(d6)), 0.2)
        reconstructed = self.deconv8(d7)  # No activation here; you can subtract from input

        return reconstructed

    def update_loss_channel(self, indices, patches, reconstructed, images, centers, patch_size=64):
      """
      Compute the loss map for each patch and update the fifth channel (history) of the corresponding images.

      Parameters:
          indices (list): List of indices indicating which images each patch corresponds to.
          patches (Tensor): Tensor of image patches (shape: [num_patches, 3, patch_size, patch_size]).
          reconstructed (Tensor): Tensor of reconstructed image patches (same shape as `patches`).
          images (Tensor): Tensor of original images (shape: [num_images, channels, height, width]).
          centers (list of tuples): List of (x, y) center coordinates for each patch.
          patch_size (int): Size of the patch (assumed to be square).

      Returns:
          loss_map (Tensor): Tensor containing the per-pixel loss map for each patch (shape: [num_patches, patch_size, patch_size]).
      """
      # Ensure input dimensions are in (batch_size, channels, height, width)
      #if x.shape[1] != 3:  # Assume input might be (batch_size, height, width, channels)
          #x = x.permute(0, 3, 1, 2)
      # Compute the loss map for each patch
      with torch.no_grad():
        loss_map = torch.mean((patches - reconstructed) ** 2, dim=1)  # Shape: [num_patches, patch_size, patch_size]

        half_patch = patch_size // 2

        # Loop through each patch and update the corresponding image
        for i, index in enumerate(indices):
            center_x, center_y = centers[i]  # Get the center for the current patch

            # Extract patch bounds
            x_start = int(center_x) - half_patch
            y_start = int(center_y) - half_patch
            x_end = x_start + patch_size
            y_end = y_start + patch_size

            # Update the fifth channel of the image at the specified index
            images[index][0][4, y_start:y_end, x_start:x_end] = loss_map[i]

# Predictor

In [ ]:
class Predictor(nn.Module):
    def __init__(self, input_channels=10):
        """
        Predictor for binary segmentation using dilated convolutions.

        Parameters:
            input_channels (int): Number of input channels (default: 10 for loss history).
        """
        super(Predictor, self).__init__()

        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1, dilation=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 16, kernel_size=3, stride=1, padding=2, dilation=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.conv3 = nn.Conv2d(16, 8, kernel_size=3, stride=1, padding=4, dilation=4)
        self.bn3 = nn.BatchNorm2d(8)
        self.conv4 = nn.Conv2d(8, 4, kernel_size=3, stride=1, padding=8, dilation=8)
        self.bn4 = nn.BatchNorm2d(4)
        self.output = nn.Conv2d(4, 1, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        """
        Forward pass for the predictor.

        Parameters:
            x (Tensor): Input loss profile tensor.

        Returns:
            Tensor: Predicted binary mask (softmax output).
        """
        x = F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.2)
        x = F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.2)
        x = F.leaky_relu(self.bn3(self.conv3(x)), negative_slope=0.2)
        x = F.leaky_relu(self.bn4(self.conv4(x)), negative_slope=0.2)
        return self.output(x)

# Define Network

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
sampler = NeuralBatchSampler(input_size=256, crop_size=128, patch_size=64, move_size=24, action_space=9).to(device)

In [ ]:
autoencoder = Autoencoder(K=200).to(device)

In [ ]:
predictor = Predictor(input_channels=10).to(device)

# Loss Function & Optimizer

In [ ]:
class CustomPredictorLoss(nn.Module):
    def __init__(self, alpha=0.5):
        """
        Custom binary segmentation loss with class balancing using weighted BCE.

        Parameters:
            alpha (float): Weighting factor to balance positive/negative classes (default: 0.5)
        """
        super(CustomPredictorLoss, self).__init__()
        self.alpha = alpha

    def forward(self, y_true, y_pred):
        """
        Forward pass for the custom loss calculation.

        Parameters:
            y_true (Tensor): Ground truth labels, shape (batch_size, 1, height, width)
            y_pred (Tensor): Predicted logits, shape (batch_size, 1, height, width)

        Returns:
            loss (Tensor): Calculated loss value.
        """
        # Apply sigmoid to get probabilities
        y_pred_sigmoid = torch.sigmoid(y_pred)

        # Compute pixel-wise binary cross-entropy loss
        pixel_loss = y_true * torch.log(y_pred_sigmoid + 1e-8) + self.alpha * (1 - y_true) * torch.log(1 - y_pred_sigmoid + 1e-8)

        # Average loss over spatial dimensions (W, H) for each image in the batch
        image_loss = torch.mean(pixel_loss, dim=(1, 2))

        # Return the mean loss across the batch
        loss = -torch.mean(image_loss)

        return loss

In [ ]:
def calculate_R_cover(input):
    """
    Compute R_cover based on the mean value of the 6th channel.
    Encourages the model to explore all regions of the image.

    Parameters:
        patches_c6: Tensor of shape [batch, height, width]
                    (stores selection frequency for each pixel)

    Returns:
        Scalar tensor representing R_cover.
    """
    if len(input) == 0:
        return torch.tensor(0.0, requires_grad=True, device=input.device)  # Ensure it is a PyTorch tensor with grad.
    return - torch.sigmoid(input.mean())

def calculate_R_clone(input):
    """
    Compute R_clone using Sobel edge detection after applying Gaussian blur on grayscale patches.
    Encourages the model to focus on edges and boundaries.

    Parameters:
        patches: Tensor of shape [batch, channels, height, width]
                 (image patches)

    Returns:
        Scalar tensor representing R_clone.
    """
    if len(input) == 0:
      return torch.tensor(0.0, requires_grad=True, device=input.device)  # Ensure it is a PyTorch tensor with grad.
    # Define Sobel filters
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=input.device).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=input.device).view(1, 1, 3, 3)

    # Convert to grayscale using weighted sum of RGB channels
    gray_patches = 0.2989 * input[:, 0:1, :, :] + 0.5870 * input[:, 1:2, :, :] + 0.1140 * input[:, 2:3, :, :]

    # Apply Gaussian Blur (define a simple 5x5 kernel)
    kernel = torch.tensor([[1, 4, 6, 4, 1], [4, 16, 24, 16, 4], [6, 24, 36, 24, 6], [4, 16, 24, 16, 4], [1, 4, 6, 4, 1]], dtype=torch.float32, device=input.device).view(1, 1, 5, 5) / 256.0
    smoothed_patch = F.conv2d(gray_patches, kernel.to(input.device), padding=2)

    # Apply Sobel filters to the blurred image
    edges_x = F.conv2d(smoothed_patch, sobel_x.to(input.device), padding=1)
    edges_y = F.conv2d(smoothed_patch, sobel_y.to(input.device), padding=1)

    # Compute edge intensity
    edge_intensity = torch.sqrt(torch.clamp(edges_x**2 + edges_y**2, min=1e-8)).mean()

    return torch.sigmoid(edge_intensity)

class RewardLoss(nn.Module):
    def __init__(self):
        """
        Reward-based loss function combining R_clone and R_cover.
        """
        super(RewardLoss, self).__init__()

    def forward(self, patches, patches_c6, R_pred, beta):
        """
        Compute R_clone and R_cover separately, then combine them.

        Parameters:
            patches: Tensor of image patches [batch, channels, height, width]
            patches_c6: Tensor representing selection frequency [batch, height, width]
            R_pred: Scalar tensor from the predictor network

        Returns:
            Negative reward (-R) as loss.
        """
        R_cover = calculate_R_cover(patches_c6)  # Compute R_cover
        R_clone = calculate_R_clone(patches)  # Compute R_clone

        # Combine rewards
        R_total = (beta * (R_clone + R_cover)) + ((1 - beta) * R_pred)

        return -R_total  # Negative reward to be used as loss

In [ ]:
# Define the Autoencoder loss (Mean Squared Error) and optimizer (Adam)
autoencoder_loss = nn.MSELoss()  # MSE loss for reconstruction
autoencoder_optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3)  # Adam optimizer for AE

# Define the Predictor loss (Binary Cross Entropy) and optimizer (Adam)
predictor_loss = CustomPredictorLoss(alpha=0.5)  # BCE loss for classification
predictor_optimizer = optim.Adam(predictor.parameters(), lr=1e-3)  # Adam optimizer for Predictor

# Define the Neural Batch Sampler loss and Optimizer (Adam)
sampler_optimizer = optim.Adam(sampler.parameters(), lr=1e-3)
reward_loss = RewardLoss()

# Train Loop

In [ ]:
buffer_capacity = 10

In [ ]:
count = sum(1 for _, ground_truth in train_data if isinstance(ground_truth, torch.Tensor) and ground_truth.dim() != 1)
h_l = [deque(maxlen=buffer_capacity) for _ in range(count)]
y_l = [ground_truth for _, ground_truth in train_data if isinstance(ground_truth, torch.Tensor) and ground_truth.dim() != 1]
countt = len(train_data) - count
h_u = [deque(maxlen=buffer_capacity) for _ in range(countt)]

In [ ]:
print(len(y_l), y_l[14].size())

45 torch.Size([1, 256, 256])


In [ ]:
print(len(h_l), len(h_u))

45 627


In [ ]:
Buffer = []

In [ ]:
alpha = 0.5
minibatch_size = 10
lowest_loss = np.inf
K, M, L = 20, 5, 100
beta = 1

In [ ]:
def reinitialize(train_data, h_l, h_u, K=200):
    #h_u = [deque([tensor.detach().cpu() for tensor in dq]) for dq in h_u]
    h_l = [deque([tensor.detach().cpu() for tensor in dq]) for dq in h_l]
    del h_l, h_u
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    count = sum(1 for _, ground_truth in train_data if isinstance(ground_truth, torch.Tensor) and ground_truth.dim() != 1)
    h_l = [deque(maxlen=buffer_capacity) for _ in range(count)]
    #y_l = [ground_truth for _, ground_truth in train_data if isinstance(ground_truth, torch.Tensor) and ground_truth.dim() != 1]
    countt = len(train_data) - count
    h_u = [deque(maxlen=buffer_capacity) for _ in range(countt)]
    autoencoder = Autoencoder(K=200).to(device)
    return h_l, h_u, autoencoder

In [ ]:
print(f"Memory usage: {psutil.virtual_memory().percent}%")

Memory usage: 33.5%


In [ ]:
"""
target_shape = torch.Size([1])
index_found = []

for idx, (image, label) in enumerate(train_data):
    if label.size() != target_shape:
        index_found.append(idx)
    if len(index_found) == 20:
      break
"""

In [ ]:
#index_found

In [ ]:
j = 0
for episode in range(100):
  patches, patches_c6, centers, index = [], [], [], []
  train_dataloader = DataLoader(train_data, batch_size=1, shuffle=False, pin_memory=True)
  for i, (image, ground_truth) in enumerate(train_dataloader):
    z = 0
    exist_flag = False
    while not exist_flag:
      if z == 0:
        patch, channel_6, center = sampler.initial_patches(image)
        center = center[0]
        patches.append(patch.squeeze(0).to(device))
        patches_c6.append(channel_6.squeeze(0).squeeze(0).to(device))
        centers.append(center)
        index.append(i)
        patch, channel_6 = patch.detach().cpu(), channel_6.detach().cpu()
        del patch, channel_6
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        #print(f'1 {i}')
        z+=1
      elif z != 0:
        #print(f'2 {i}')
        crop, center_crop = sampler.select_crops(image, center)
        with torch.no_grad():
          reconstructed_crop = autoencoder.forward(crop[:, :3, :, :].to(device))
        #print(crop[:, :3, :, :].size(), reconstructed_crop.size())
        crop = crop.clone().to(device)
        crop[:, 4, :, :] = torch.mean((crop[:, :3, :, :] - reconstructed_crop.detach()) ** 2, dim=1)
        #print(crop.size())
        action_probs = sampler.forward(crop)
        action = torch.multinomial(action_probs, 1)
        if action == 0:
          #print('change state')
          #print(len(patches))
          exist_flag = True
        else:
          patch, channel_6, center = sampler.move_select_patches(image=image, action=int(action[0].item()), center=center_crop)
          patches.append(patch.squeeze(0).to(device))
          patches_c6.append(channel_6.squeeze(0).to(device))
          centers.append(center)
          index.append(i)
          patch, channel_6 = patch.detach().cpu(), channel_6.detach().cpu()
          del patch, channel_6
          gc.collect()
          torch.cuda.empty_cache()
          torch.cuda.ipc_collect()
        crop, reconstructed_crop, action, action_probs = crop.detach().cpu(), reconstructed_crop.detach().cpu(), action.detach().cpu(), action_probs.detach().cpu()
        del crop, reconstructed_crop, action, action_probs
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
      if (len(patches) == minibatch_size) or (len(patches) == minibatch_size - 1):

        patches_c6 = torch.stack(patches_c6).to(device)
        patches_c6_1 = patches_c6.clone().to(device)
        patches = torch.stack(patches).to(device)
        patches_1 = patches.clone().to(device)
        autoencoder_optimizer.zero_grad() # remove gradient
        reconstructeds = autoencoder.forward(patches)
        #print(reconstructeds.grad_fn)
        #print(reconstructeds.size())
        loss_ae = autoencoder_loss(reconstructeds, patches)
        loss_ae.backward() # Backpropagation
        autoencoder_optimizer.step() # update weight AE

        # Move data to CPU to free GPU memory
        reconstructeds = reconstructeds.detach().cpu()
        patches = patches.detach().cpu()
        patches_c6 = patches_c6.detach().cpu()
        print(f'Epoch: {episode + 1}, Step: {j + 1}, loss AutoEncoder: {loss_ae}')

        autoencoder.update_loss_channel(indices=index, patches=patches, reconstructed=reconstructeds, images=train_data, centers=centers)
        del patches, patches_c6, centers, index
        patches = []
        patches_c6 = []
        #print(len(patches))
        centers = []
        index = []
        del reconstructeds, loss_ae
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        autoencoder.eval()
        with torch.no_grad():
          l, u = 0, 0
          for ind, (imag, labe) in enumerate(train_data):
            #print(image.unsqueeze(0).size())
            imag = imag.to(device)
            reconstructed = autoencoder.forward(imag.unsqueeze(0)[:, :3, :, :])
            #print(reconstructed.size())
            #error =
            #print(error.squeeze(0).size())
            if isinstance(labe, torch.Tensor) and labe.dim() != 1:
              h_l[l].append(torch.mean((imag.unsqueeze(0)[:, :3, :, :] - reconstructed) ** 2, dim=1).squeeze(0).cpu())
              l+=1
            #elif isinstance(labe, torch.Tensor) and labe.dim() == 1:
              #h_u[u].append(error.squeeze(0).cpu())
              #u+=1

            del imag, labe, reconstructed
            torch.cuda.empty_cache()
            if ind % 100 == 0:
              gc.collect()
              torch.cuda.empty_cache()
              torch.cuda.ipc_collect()

        gc.collect()
        os.system('echo 1 > /proc/sys/vm/drop_caches')
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        autoencoder.train()

        if all(len(deq) == buffer_capacity for deq in h_l):
          Buffer.extend([(torch.stack([t.to(device) for t in deq]), labels) for deq, labels in zip(h_l, y_l)])

          num_samples = 20
          num_epochs = 5

          for epoch in range(num_epochs):
              sampled_indices = torch.randperm(len(Buffer))[:num_samples]
              sampled_data = [Buffer[i] for i in sampled_indices]
              loss_profiles_sample = [sampled_data[i][0] for i in range(num_samples)]
              labels_samples = [sampled_data[i][1] for i in range(num_samples)]
              sampled_data = [(tensor1.to('cpu'), tensor2.to('cpu')) for tensor1, tensor2 in sampled_data]
              sampled_indices = [tensor.detach().cpu() for tensor in sampled_indices]
              del sampled_data, sampled_indices
              gc.collect()
              torch.cuda.empty_cache()
              torch.cuda.ipc_collect()

              loss_profiles_sample = torch.stack(loss_profiles_sample).to(device)
              labels_samples = torch.stack(labels_samples).to(device)

              predictor_optimizer.zero_grad() # Clear previous gradients

              prediction = predictor(loss_profiles_sample) # Forward pass through the predicton

              loss_p = predictor_loss(labels_samples.squeeze(), prediction.squeeze())  # use the correct loss function

              loss_p.backward() # Backpropagation
              predictor_optimizer.step() # Update model parameters
              del prediction
              gc.collect()
              torch.cuda.empty_cache()
              torch.cuda.ipc_collect()
              print(f'Epoch: {episode + 1}, Step: {j + 1}, loss Predicton: {loss_p};           Epoch for Predictor:{epoch + 1}')

          del Buffer
          gc.collect()
          os.system('echo 1 > /proc/sys/vm/drop_caches')
          torch.cuda.empty_cache()
          torch.cuda.ipc_collect()
          Buffer = []
          #if loss_p.detach().cpu().numpy() < lowest_loss:
            #print('Updata h_star & lowest loss')
            #h_star = h_u
            #lowest_loss = loss_p.detach().cpu().numpy()

        #if j%k>M: calculate R_Pred and update NBS
        if ((j % K) > M) and (all(len(deq) == buffer_capacity for deq in h_l)):

          sampler_optimizer.zero_grad()
          # Detach to prevent gradients flowing back to Predictor
          detached_loss_profiles_sample = loss_profiles_sample.detach().to(device)
          detached_prediction = predictor(detached_loss_profiles_sample)
          detached_loss_p = predictor_loss(labels_samples.squeeze(), detached_prediction.squeeze())
          #sampler_optimizer.zero_grad() # NBS Optimizer zero garad
          R_pred = - detached_loss_p

          # Compute loss
          loss_nbs = reward_loss(patches_1, patches_c6_1, R_pred, beta)

          # Backpropagation
          loss_nbs.backward()
          sampler_optimizer.step()

          print(f'Epoch: {episode + 1}, Step: {j + 1}, loss NBS: {loss_nbs}')

          detached_loss_profiles_sample = detached_loss_profiles_sample.detach().cpu()
          detached_prediction = detached_prediction.detach().cpu()
          patches_1 = patches_1.detach().cpu()
          patches_c6_1 = patches_c6_1.detach().cpu()
          loss_profiles_sample = loss_profiles_sample.detach().cpu()
          labels_samples = labels_samples.detach().cpu()
          del R_pred, loss_nbs
          del detached_loss_profiles_sample, detached_prediction, patches_c6_1, patches_1, loss_profiles_sample, labels_samples
          gc.collect()
          torch.cuda.empty_cache()
          torch.cuda.ipc_collect()

        if ((j % K) == 0) and (j != 0):
          print(f'j: {j};   ReInitial AutoEncoder & h_l, h_u')
          h_l, h_u, autoencoder = reinitialize(train_data, h_l, h_u)

        # Update j and beta
        beta = max(0, 1 - (j / L))
        print(f'Update Beta: {beta}, j: {j}')
        j += 1

        gc.collect()
        os.system('echo 1 > /proc/sys/vm/drop_caches')
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

Epoch: 1, Step: 1, loss AutoEncoder: 0.3059888184070587
Update Beta: 1.0, j: 0
Epoch: 1, Step: 2, loss AutoEncoder: 0.9913735389709473
Update Beta: 0.99, j: 1
Epoch: 1, Step: 3, loss AutoEncoder: 1.0448704957962036
Update Beta: 0.98, j: 2
Epoch: 1, Step: 4, loss AutoEncoder: 0.9707748889923096
Update Beta: 0.97, j: 3
Epoch: 1, Step: 5, loss AutoEncoder: 0.9946311712265015
Update Beta: 0.96, j: 4
Epoch: 1, Step: 6, loss AutoEncoder: 1.009251594543457
Update Beta: 0.95, j: 5
Epoch: 1, Step: 7, loss AutoEncoder: 0.9660077095031738
Update Beta: 0.94, j: 6
Epoch: 1, Step: 8, loss AutoEncoder: 1.0872819423675537
Update Beta: 0.9299999999999999, j: 7
Epoch: 1, Step: 9, loss AutoEncoder: 1.0925763845443726
Update Beta: 0.92, j: 8
Epoch: 1, Step: 10, loss AutoEncoder: 1.005997896194458
Epoch: 1, Step: 10, loss Predicton: 0.452309250831604;           Epoch for Predictor:1
Epoch: 1, Step: 10, loss Predicton: 0.4523285925388336;           Epoch for Predictor:2
Epoch: 1, Step: 10, loss Predicton: 0

In [ ]:
# Save Models
torch.save(sampler.state_dict(), 'sampler_model.pth')
torch.save(predictor.state_dict(), 'predictor_model.pth')

In [ ]:
sampler_test = NeuralBatchSampler(input_size=256, crop_size=128, patch_size=64, move_size=24, action_space=9).to(device)
predictor_test = Predictor(input_channels=10).to(device)

In [ ]:
# Load Models
sampler_test.load_state_dict(torch.load('sampler_model.pth'))
predictor_test.load_state_dict(torch.load('predictor_model.pth'))

<All keys matched successfully>

In [ ]:
sampler_test.eval()
predictor_test.eval()

Predictor(
  (conv1): Conv2d(10, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2))
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(4, 4), dilation=(4, 4))
  (bn3): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(8, 4, kernel_size=(3, 3), stride=(1, 1), padding=(8, 8), dilation=(8, 8))
  (bn4): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (output): Conv2d(4, 1, kernel_size=(1, 1), stride=(1, 1))
)

In [ ]:
for param in sampler.parameters():
    param.requires_grad = False
for param in predictor.parameters():
    param.requires_grad = False

In [ ]:
sampler_input = torch.randn(9, 6, 128, 128).to('cuda')
predictor_input = torch.randn(9, 10, 256, 256).to('cuda')

In [ ]:
with torch.no_grad():
  x=predictor_test.forward(predictor_input)

In [ ]:
x.size()

torch.Size([9, 1, 256, 256])

In [ ]:
with torch.no_grad():
  y=sampler_test.forward(sampler_input)

In [ ]:
y.size()

torch.Size([9, 9])

In [ ]:
y

tensor([[6.4935e-02, 9.5617e-04, 1.3571e-01, 1.9075e-02, 2.1956e-02, 5.3048e-04,
         7.2645e-01, 8.8949e-03, 2.1498e-02],
        [1.0215e-01, 2.0174e-03, 5.4867e-01, 1.6496e-02, 6.1283e-02, 2.6884e-02,
         2.2448e-01, 4.2525e-03, 1.3763e-02],
        [1.7061e-01, 4.6414e-03, 3.6113e-01, 9.5663e-02, 2.2939e-02, 2.0044e-01,
         1.1524e-01, 4.5881e-03, 2.4750e-02],
        [1.1972e-03, 2.4188e-02, 5.6304e-02, 8.6961e-03, 4.1385e-02, 5.4403e-04,
         4.5803e-01, 1.7385e-03, 4.0792e-01],
        [1.5228e-01, 5.8156e-03, 1.4504e-01, 3.3970e-02, 4.1182e-01, 5.4763e-03,
         1.7679e-01, 4.4644e-03, 6.4344e-02],
        [2.6021e-01, 9.1968e-03, 3.1767e-01, 1.3284e-01, 5.1902e-02, 6.3915e-04,
         1.8407e-01, 3.2368e-02, 1.1109e-02],
        [6.0958e-01, 6.9120e-03, 1.2602e-01, 9.9048e-03, 1.6465e-02, 9.5721e-03,
         1.8243e-01, 1.0150e-02, 2.8977e-02],
        [1.5252e-02, 3.3782e-03, 1.3180e-01, 9.3437e-03, 2.6713e-02, 2.1978e-03,
         5.7771e-01, 5.3250e-0